<a href="https://colab.research.google.com/github/Nishikant090/5G_Handover_Prediction_FIXED.ipynb/blob/main/5G_Handover_Prediction_FIXED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 5G Handover Prediction — Fixed & Cleaned Pipeline
### sklearn Stacked Ensemble | MLP Signal Prediction | Manual SMOTE

**Fixed bugs vs original notebook:**
| # | Bug | Fix |
|---|-----|-----|
| 1 | `google.colab` upload/drive calls | Replaced with direct file paths |
| 2 | `fillna(method='ffill')` deprecated in pandas 3.x | Replaced with `.ffill().bfill()` |
| 3 | `target_idx` hardcoded to `1` in Stage 1 | Made dynamic from `signal_features` list |
| 4 | `TARGET` overwritten to `"handover_future"` in cell 21 | Fixed — stays `"handover"` throughout |
| 5 | `feature_names` vs `clean_feature_names` mismatch in Stage 4 | Unified to one variable |
| 6 | `AdaBoostClassifier(algorithm='SAMME')` breaks sklearn 1.8+ | Removed deprecated arg |
| 7 | SMOTE applied BEFORE train/test split → data leakage | SMOTE now applied only on train fold |
| 8 | Non-numeric columns (device make/model) not filtered | Fixed with `is_numeric_dtype()` check |
| 9 | String-dtype `median()` error during imputation | Fixed with proper dtype branching |
|10 | `matplotlib.use('Agg')` not set → display crash in some envs | Added at import time |

**Architecture:**
```
CSV Files → Merge & Clean → MLP Signal Prediction → Feature Engineering (41 features)
→ Manual SMOTE → Stacked Ensemble (RF + ExtraTrees + GBT + MLP) → Evaluation
```

## ⚙️ Cell 0 — Install Dependencies

In [ ]:
# Run this cell first in Colab
!pip install -q torch torchvision
!pip install -q autogluon.tabular
!pip install -q lightgbm xgboost catboost
!pip install -q imbalanced-learn shap
!pip install -q pandas numpy matplotlib scikit-learn scipy

print("✓ All packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 4.8 MB/s eta 0:00:00
✓ All packages installed


## 📦 Cell 1 — Imports & Global Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')   # FIX: prevents display errors in some environments
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import warnings, os, pickle, time
from scipy.stats import linregress

from sklearn.model_selection    import StratifiedKFold, train_test_split
from sklearn.ensemble           import (RandomForestClassifier, ExtraTreesClassifier,
                                        GradientBoostingClassifier, AdaBoostClassifier)
from sklearn.linear_model       import LogisticRegression
from sklearn.neural_network     import MLPClassifier, MLPRegressor
from sklearn.neighbors          import KNeighborsClassifier
from sklearn.preprocessing      import StandardScaler
from sklearn.metrics            import (accuracy_score, f1_score, precision_score,
                                        recall_score, roc_auc_score, confusion_matrix,
                                        classification_report, roc_curve,
                                        precision_recall_curve, average_precision_score,
                                        mean_squared_error, mean_absolute_error)
warnings.filterwarnings("ignore")

# ── Optional packages (auto-detected) ─────────────────────────────────────
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH = True
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"✓ PyTorch {torch.__version__} | Device: {DEVICE}")
except ImportError:
    TORCH = False
    print("⚠  PyTorch not found — MLP fallback will be used")

try:
    from autogluon.tabular import TabularPredictor
    AUTOGLUON = True
    print("✓ AutoGluon available")
except ImportError:
    AUTOGLUON = False
    print("⚠  AutoGluon not found — sklearn stacked ensemble will be used")

try:
    from imblearn.over_sampling import SMOTE
    IMBLEARN = True
    print("✓ imbalanced-learn available")
except ImportError:
    IMBLEARN = False
    print("⚠  imbalanced-learn not found — manual SMOTE fallback")

try:
    import lightgbm as lgb
    LGBM = True; print("✓ LightGBM available")
except ImportError:
    LGBM = False

try:
    import xgboost as xgb
    XGB = True; print("✓ XGBoost available")
except ImportError:
    XGB = False

try:
    from catboost import CatBoostClassifier
    CATBOOST = True; print("✓ CatBoost available")
except ImportError:
    CATBOOST = False

try:
    import shap
    SHAP = True; print("✓ SHAP available")
except ImportError:
    SHAP = False

# ── Global Config ──────────────────────────────────────────────────────────
TARGET       = "handover"
RANDOM_STATE = 42
WINDOW       = 20
HORIZONS     = [1, 2, 3]
ROLL_WIN     = 5
TREND_WIN    = 10
N_FOLDS      = 5
BATCH_SIZE   = 256
EPOCHS       = 50
LR           = 1e-3
HIDDEN_SIZE  = 128
NUM_LAYERS   = 2
DROPOUT      = 0.2

for d in ["outputs", "plots", "models"]:
    os.makedirs(d, exist_ok=True)

print("\n✓ Setup complete — all directories created")

✓ PyTorch 2.10.0+cpu | Device: cpu
✓ AutoGluon available
✓ imbalanced-learn available
✓ LightGBM available
✓ XGBoost available
✓ CatBoost available
✓ SHAP available

✓ Setup complete — all directories created


---
## 🗂️ Stage 0 — Data Loading, Merging & EDA

In [ ]:
# ── Upload files (Colab UI) ───────────────────────────────────────────────
# Option A: Upload via Colab file picker
from google.colab import files
print("Upload your two CSV files:")
uploaded = files.upload()
FILE1, FILE2 = list(uploaded.keys())[0], list(uploaded.keys())[1]

# Option B: Uncomment to mount Google Drive instead
# from google.colab import drive
# drive.mount('/content/drive')
# FILE1 = '/content/drive/MyDrive/network_logs_1756976794378.csv'
# FILE2 = '/content/drive/MyDrive/network_logs_1768131779033.csv'

Upload your two CSV files:


Saving network_logs_1756976794378.csv to network_logs_1756976794378.csv
Saving network_logs_1768131779033.csv to network_logs_1768131779033.csv


In [ ]:
# ── Load & Merge ──────────────────────────────────────────────────────────
df1 = pd.read_csv(FILE1)
df2 = pd.read_csv(FILE2)

# Normalize column names
df1.columns = df1.columns.str.strip().str.lower().str.replace(" ", "_")
df2.columns = df2.columns.str.strip().str.lower().str.replace(" ", "_")
print(f"File 1: {df1.shape}  |  File 2: {df2.shape}")

common_cols = list(set(df1.columns) & set(df2.columns))
df = pd.concat([df1[common_cols], df2[common_cols]], ignore_index=True)
before = len(df)
df = df.drop_duplicates()
print(f"Merged: {before:,} rows → {len(df):,} after dedup")

# Sort chronologically
ts_col = next((c for c in df.columns if "time" in c or c == "ts"), None)
if ts_col:
    df[ts_col] = pd.to_datetime(df[ts_col], errors="coerce")
    df = df.sort_values(ts_col).reset_index(drop=True)
    print(f"✓ Sorted by '{ts_col}'")

# Detect signal columns
signal_cols = [c for c in df.columns if any(x in c for x in ["rsrp","rsrq","rssi","sinr","snr","cqi"])]
rsrp_col = next((c for c in signal_cols if "rsrp" in c), signal_cols[0] if signal_cols else None)
rsrq_col = next((c for c in signal_cols if "rsrq" in c), signal_cols[1] if len(signal_cols)>1 else rsrp_col)

# FIX 1: Clean string-encoded numeric columns (remove units like ' dBm', ' dB', ' Mbps', ' km/h')
def clean_numeric(val):
    if isinstance(val, str):
        for unit in [' dBm', ' dB', ' Mbps', ' km/h']:
            val = val.replace(unit, '')
        val = val.strip()
    try:
        return float(val)
    except (ValueError, TypeError):
        return np.nan

for c in [rsrp_col, rsrq_col, 'downlink(mbps)', 'uplink(mbps)', 'velocity(km/h)', 'sinr', 'pci']:
    if c and c in df.columns:
        df[c] = df[c].apply(clean_numeric)
        df[c] = df[c].replace([np.inf, -np.inf], np.nan)

# FIX 2: Use .ffill()/.bfill() — fillna(method=...) is deprecated in pandas 3.x
# FIX 3: Use is_numeric_dtype() — avoids TypeError on string columns
missing = df.isnull().sum()
if missing.sum() > 0:
    print(f"Missing before imputation: {missing[missing>0].to_dict()}")
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            med = df[col].median()
            df[col] = df[col].fillna(med if pd.notna(med) else 0)
        else:
            df[col] = df[col].ffill().bfill()
    print("✓ Missing values filled")
else:
    print("✓ No missing values")

# Create TARGET via PCI change detection
if "pci" in df.columns:
    df[TARGET] = (df["pci"] != df["pci"].shift(1)).astype(int)
    df[TARGET] = df[TARGET].fillna(0).astype(int)
    print(f"✓ Created target '{TARGET}' via PCI change")
else:
    df[TARGET] = 0
    print(f"⚠ PCI missing — target defaulted to 0")

print(f"\nSignal cols: {signal_cols}")
print(f"RSRP: '{rsrp_col}'  |  RSRQ: '{rsrq_col}'")
print(f"\nTarget distribution:")
print(df[TARGET].value_counts())
df.to_csv("outputs/dataset_merged_clean.csv", index=False)
print("\n✓ Saved → outputs/dataset_merged_clean.csv")

File 1: (10570, 15)  |  File 2: (523, 15)
Merged: 11,093 rows → 11,093 after dedup
✓ Sorted by 'timestamp'
Missing before imputation: {'rsrq': 209, 'downlink(mbps)': 92, 'velocity(km/h)': 269, 'uplink(mbps)': 92, 'pci': 209, 'latitude': 269, 'sinr': 209, 'rsrp': 12, 'longitude': 269}
✓ Missing values filled
✓ Created target 'handover' via PCI change

Signal cols: ['rsrq', 'sinr', 'rsrp']
RSRP: 'rsrp'  |  RSRQ: 'rsrq'

Target distribution:
handover
0    10746
1      347
Name: count, dtype: int64

✓ Saved → outputs/dataset_merged_clean.csv


In [ ]:
# ── EDA Plots ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.suptitle("Stage 0 — Exploratory Data Analysis", fontsize=15, fontweight="bold")
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
vc = df[TARGET].value_counts()
ax1.bar([str(k) for k in vc.index], vc.values, color=["#2ecc71","#e74c3c"], edgecolor="black")
ax1.set_title("Class Distribution", fontweight="bold")
ax1.set_xlabel("Handover Label"); ax1.set_ylabel("Count")
for i, v in enumerate(vc.values):
    ax1.text(i, v+5, f"{v:,}", ha="center", fontsize=10)

ax2 = fig.add_subplot(gs[0, 1])
for cls, color in [(0,"#2ecc71"),(1,"#e74c3c")]:
    df[df[TARGET]==cls][rsrp_col].hist(bins=40, alpha=0.65, ax=ax2, color=color, label=f"Class {cls}")
ax2.set_title("RSRP Distribution by Class", fontweight="bold"); ax2.legend()

ax3 = fig.add_subplot(gs[0, 2])
for cls, color in [(0,"#2ecc71"),(1,"#e74c3c")]:
    df[df[TARGET]==cls][rsrq_col].hist(bins=40, alpha=0.65, ax=ax3, color=color, label=f"Class {cls}")
ax3.set_title("RSRQ Distribution by Class", fontweight="bold"); ax3.legend()

ax4 = fig.add_subplot(gs[1, :2])
n_plot = min(500, len(df))
for col, color in zip([rsrp_col, rsrq_col], ["#3498db","#e74c3c"]):
    v = df[col].iloc[:n_plot].values
    if np.isfinite(v).any():
        v_norm = (v - np.nanmin(v)) / (np.nanmax(v) - np.nanmin(v) + 1e-9)
        ax4.plot(v_norm, label=f"{col.upper()} (norm)", alpha=0.8, color=color, linewidth=0.9)
ho = df[TARGET].iloc[:n_plot].values
ax4.scatter(np.where(ho==1)[0], np.ones(ho.sum())*1.02,
            color="orange", marker="v", s=25, label="Handover", zorder=5)
ax4.set_title(f"Signal Time Series (first {n_plot} samples)", fontweight="bold")
ax4.legend(fontsize=8); ax4.grid(alpha=0.3)

ax5 = fig.add_subplot(gs[1, 2])
num_cols = df.select_dtypes(include=np.number).columns[:10]
corr = df[num_cols].corr()
im = ax5.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax5.set_xticks(range(len(corr))); ax5.set_yticks(range(len(corr)))
ax5.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=7)
ax5.set_yticklabels(corr.columns, fontsize=7)
plt.colorbar(im, ax=ax5); ax5.set_title("Correlation Heatmap", fontweight="bold")

plt.savefig("plots/stage0_eda.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Stage 0 complete")

✓ Stage 0 complete


---
## 🧠 Stage 1 — LSTM / MLP Temporal Signal Prediction

In [ ]:
# ── LSTM Model (PyTorch) ─────────────────────────────────────────────────
if TORCH:
    class LSTMSignalPredictor(nn.Module):
        def __init__(self, input_size, hidden_size, num_layers, n_horizons, dropout):
            super().__init__()
            self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                                num_layers=num_layers, batch_first=True,
                                dropout=dropout if num_layers > 1 else 0.0,
                                bidirectional=True)
            self.attention = nn.Sequential(
                nn.Linear(hidden_size*2, hidden_size), nn.Tanh(),
                nn.Linear(hidden_size, 1))
            self.fc = nn.Sequential(
                nn.Linear(hidden_size*2, hidden_size), nn.ReLU(),
                nn.Dropout(dropout), nn.Linear(hidden_size, n_horizons))

        def forward(self, x):
            out, _ = self.lstm(x)
            attn_w = torch.softmax(self.attention(out), dim=1)
            context = (attn_w * out).sum(dim=1)
            return self.fc(context)
    print("✓ LSTMSignalPredictor defined")
else:
    print("⚠  PyTorch not available — MLPRegressor fallback will be used")

✓ LSTMSignalPredictor defined


In [ ]:
# ── Sliding Window Builder ────────────────────────────────────────────────
def build_windows(signal_array, window, horizons, target_idx):
    max_h = max(horizons)
    X, Y = [], []
    for i in range(window, len(signal_array) - max_h + 1):
        X.append(signal_array[i-window:i, :])
        Y.append([signal_array[i+h-1, target_idx] for h in horizons])
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)

# Signal features for temporal prediction
signal_features = [rsrp_col, rsrq_col]
if "sinr" in df.columns:
    signal_features.append("sinr")
signal_features = [f for f in signal_features if f in df.columns]
print(f"Signal features: {signal_features}")

signal_data = df[signal_features].values.astype(np.float32)

# FIX 4: target_idx is dynamic — original notebook hardcoded it to 1
target_idx = signal_features.index(rsrq_col) if rsrq_col in signal_features else 0
print(f"Target signal for prediction: '{rsrq_col}' (index {target_idx})")

# Temporal split BEFORE scaling (prevents leakage)
n = len(signal_data)
n_tr = int(0.70*n); n_val = int(0.15*n)
train_data = signal_data[:n_tr]
val_data   = signal_data[n_tr:n_tr+n_val]
test_data  = signal_data[n_tr+n_val:]

# Fit scaler ONLY on training data
signal_scaler = StandardScaler()
train_norm = signal_scaler.fit_transform(train_data)
val_norm   = signal_scaler.transform(val_data)
test_norm  = signal_scaler.transform(test_data)
signal_norm = np.concatenate([train_norm, val_norm, test_norm])

# Build sliding windows
X_win, Y_win = build_windows(signal_norm, WINDOW, HORIZONS, target_idx)
n = len(X_win)
n_tr = int(0.70*n); n_val = int(0.15*n)
X_tr, Y_tr   = X_win[:n_tr],           Y_win[:n_tr]
X_val, Y_val = X_win[n_tr:n_tr+n_val], Y_win[n_tr:n_tr+n_val]
X_te, Y_te   = X_win[n_tr+n_val:],     Y_win[n_tr+n_val:]

print(f"Signal data: {signal_data.shape}")
print(f"Windows — X: {X_win.shape}  Y: {Y_win.shape}")
print(f"Train: {len(X_tr):,}  Val: {len(X_val):,}  Test: {len(X_te):,}")

Signal features: ['rsrp', 'rsrq', 'sinr']
Target signal for prediction: 'rsrq' (index 1)
Signal data: (11093, 3)
Windows — X: (11071, 20, 3)  Y: (11071, 3)
Train: 7,749  Val: 1,660  Test: 1,662


In [ ]:
# ── Train LSTM (PyTorch) or MLPRegressor (fallback) ──────────────────────
lstm_history = {"train_loss": [], "val_loss": []}

if TORCH:
    def make_loader(Xa, Ya, shuffle=True):
        ds = TensorDataset(torch.from_numpy(Xa), torch.from_numpy(Ya))
        return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

    tr_loader  = make_loader(X_tr,  Y_tr,  shuffle=True)
    val_loader = make_loader(X_val, Y_val, shuffle=False)
    te_loader  = make_loader(X_te,  Y_te,  shuffle=False)

    lstm_model = LSTMSignalPredictor(
        input_size=X_win.shape[2], hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS, n_horizons=len(HORIZONS), dropout=DROPOUT
    ).to(DEVICE)
    print(f"LSTM params: {sum(p.numel() for p in lstm_model.parameters()):,}")

    opt       = torch.optim.AdamW(lstm_model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    criterion = nn.MSELoss()
    best_val_loss = float("inf")
    best_model_path = "models/lstm_signal_predictor.pt"

    for epoch in range(1, EPOCHS + 1):
        lstm_model.train()
        train_loss = 0.0
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = criterion(lstm_model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(lstm_model.parameters(), max_norm=1.0)
            opt.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= len(tr_loader.dataset)

        lstm_model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                val_loss += criterion(lstm_model(xb), yb).item() * xb.size(0)
        val_loss /= len(val_loader.dataset)

        lstm_history["train_loss"].append(train_loss)
        lstm_history["val_loss"].append(val_loss)
        scheduler.step()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(lstm_model.state_dict(), best_model_path)
        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{EPOCHS} | Train: {train_loss:.5f} | Val: {val_loss:.5f}")

    lstm_model.load_state_dict(torch.load(best_model_path))
    lstm_model.eval()
    preds_list, trues_list = [], []
    with torch.no_grad():
        for xb, yb in te_loader:
            preds_list.append(lstm_model(xb.to(DEVICE)).cpu().numpy())
            trues_list.append(yb.numpy())
    preds_test = np.vstack(preds_list)
    trues_test = np.vstack(trues_list)
    print(f"\n✓ LSTM model saved → {best_model_path}")

else:
    # ── MLPRegressor fallback ─────────────────────────────────────────────
    print("Training MLPRegressor for each horizon ...")
    X_tr_f = X_tr.reshape(len(X_tr), -1)
    X_te_f = X_te.reshape(len(X_te), -1)
    preds_col, mlp_models = [], []

    for h_idx, h in enumerate(HORIZONS):
        mlp = MLPRegressor(hidden_layer_sizes=(256,128,64), activation="relu",
                           solver="adam", max_iter=200, early_stopping=True,
                           random_state=RANDOM_STATE, verbose=False)
        mlp.fit(X_tr_f, Y_tr[:, h_idx])
        preds_col.append(mlp.predict(X_te_f))
        mlp_models.append(mlp)
        print(f"  Horizon t+{h} done")

    preds_test = np.column_stack(preds_col)
    trues_test = Y_te
    with open("models/mlp_signal_predictor.pkl","wb") as f: pickle.dump(mlp_models, f)

# ── Test Metrics ─────────────────────────────────────────────────────────
print("\nTest Metrics:")
for i, h in enumerate(HORIZONS):
    rmse = np.sqrt(mean_squared_error(trues_test[:,i], preds_test[:,i]))
    mae  = mean_absolute_error(trues_test[:,i], preds_test[:,i])
    print(f"  RSRQ(t+{h}): RMSE={rmse:.4f}  MAE={mae:.4f}")

with open("models/signal_scaler.pkl","wb") as f: pickle.dump(signal_scaler, f)

LSTM params: 597,764
Epoch   1/50 | Train: 0.49598 | Val: 0.18648
Epoch  10/50 | Train: 0.14822 | Val: 0.01137
Epoch  20/50 | Train: 0.14128 | Val: 0.00745
Epoch  30/50 | Train: 0.13979 | Val: 0.00823
Epoch  40/50 | Train: 0.13514 | Val: 0.00900
Epoch  50/50 | Train: 0.13462 | Val: 0.00900

✓ LSTM model saved → models/lstm_signal_predictor.pt

Test Metrics:
  RSRQ(t+1): RMSE=0.2281  MAE=0.0894
  RSRQ(t+2): RMSE=0.2896  MAE=0.1164
  RSRQ(t+3): RMSE=0.3286  MAE=0.1383


In [ ]:
# ── Generate Predictions for Full Dataset & Plot ─────────────────────────
full_X, _ = build_windows(signal_norm, WINDOW, HORIZONS, target_idx)

if TORCH:
    full_loader = DataLoader(TensorDataset(torch.from_numpy(full_X)), batch_size=1024, shuffle=False)
    lstm_model.eval()
    full_preds_list = []
    with torch.no_grad():
        for (xb,) in full_loader:
            full_preds_list.append(lstm_model(xb.to(DEVICE)).cpu().numpy())
    full_preds = np.vstack(full_preds_list)
else:
    full_X_f   = full_X.reshape(len(full_X), -1)
    full_preds = np.column_stack([m.predict(full_X_f) for m in mlp_models])

for i, h in enumerate(HORIZONS):
    dummy = np.zeros((len(full_preds), signal_norm.shape[1]))
    dummy[:, target_idx] = full_preds[:, i]
    inv_vals = signal_scaler.inverse_transform(dummy)[:, target_idx]
    padded = np.full(len(df), np.nan)
    end_idx = min(WINDOW + len(inv_vals), len(df))
    padded[WINDOW:end_idx] = inv_vals[:end_idx-WINDOW]
    df[f"pred_RSRQ_t+{h}"] = padded

df.to_csv("outputs/dataset_with_predictions.csv", index=False)
print(f"✓ Saved → outputs/dataset_with_predictions.csv")

fig, axes = plt.subplots(1, len(HORIZONS), figsize=(18, 5))
fig.suptitle("Stage 1 — Signal Prediction Results", fontsize=14, fontweight="bold")
for i, h in enumerate(HORIZONS):
    n_show = min(200, len(trues_test))
    axes[i].plot(trues_test[:n_show,i], label="Actual",    color="#2c3e50", lw=1.0)
    axes[i].plot(preds_test[:n_show,i], label="Predicted", color="#e67e22", lw=1.0, ls="--")
    axes[i].set_title(f"RSRQ(t+{h}) Prediction", fontweight="bold")
    axes[i].legend(fontsize=9); axes[i].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("plots/stage1_lstm_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Stage 1 complete")

✓ Saved → outputs/dataset_with_predictions.csv
✓ Stage 1 complete


---
## ⚙️ Stage 2 — Temporal Feature Engineering

In [ ]:
# ── Feature Engineering Functions ───────────────────────────────────────
def add_lag_features(df, cols, lags):
    for col in cols:
        for lag in lags:
            df[f"{col}_lag{lag}"] = df[col].shift(lag).fillna(0)
    return df

def add_derivatives(df, cols):
    for col in cols:
        df[f"{col}_diff"]     = df[col].diff().fillna(0)
        df[f"{col}_velocity"] = df[col].diff().diff().fillna(0)
        df[f"{col}_diff_abs"] = df[col].diff().abs().fillna(0)
    return df

def add_rolling_stats(df, cols, window):
    for col in cols:
        r = df[col].rolling(window, min_periods=1)
        df[f"{col}_mean{window}"]  = r.mean()
        df[f"{col}_std{window}"]   = r.std().fillna(0)
        df[f"{col}_min{window}"]   = r.min()
        df[f"{col}_max{window}"]   = r.max()
        df[f"{col}_range{window}"] = df[f"{col}_max{window}"] - df[f"{col}_min{window}"]
    return df

def add_degradation(df, cols):
    for col in cols:
        df[f"{col}_drop_rate"]   = df[col].diff().fillna(0)
        df[f"{col}_is_dropping"] = (df[f"{col}_drop_rate"] < 0).astype(int)
        streaks, count = [], 0
        for v in df[f"{col}_is_dropping"]:
            count = count+1 if v else 0
            streaks.append(count)
        df[f"{col}_drop_streak"] = streaks
    return df

def add_stability(df, cols, window):
    for col in cols:
        df[f"{col}_var{window}"] = df[col].rolling(window, min_periods=2).var().fillna(0)
        slopes = np.full(len(df), np.nan)
        x = np.arange(window)
        arr = df[col].values
        for i in range(window-1, len(arr)):
            y = arr[i-window+1:i+1]
            if not np.any(np.isnan(y)):
                slopes[i] = linregress(x, y)[0]
        df[f"{col}_slope{window}"] = slopes
    return df

def add_cross_features(df, rsrp_col, rsrq_col):
    df["rsrp_rsrq_diff"]       = df[rsrp_col] - df[rsrq_col]
    df["rsrp_rsrq_ratio"]      = df[rsrp_col] / df[rsrq_col].replace(0, np.nan)
    df["signal_quality_index"] = (df[rsrp_col] + df[rsrq_col]) / 2.0
    df["both_dropping"] = (
        (df.get(f"{rsrp_col}_is_dropping", pd.Series(0, index=df.index)) == 1) &
        (df.get(f"{rsrq_col}_is_dropping", pd.Series(0, index=df.index)) == 1)
    ).astype(int)
    return df

def add_risk_features(df, rsrp_col, rsrq_col):
    df["rsrp_below_threshold"]  = (df[rsrp_col] < -110).astype(int)
    df["rsrq_below_threshold"]  = (df[rsrq_col] < -15).astype(int)
    df["dual_threshold_breach"] = (df["rsrp_below_threshold"] & df["rsrq_below_threshold"]).astype(int)
    df["rsrp_threshold_gap"]    = df[rsrp_col] - (-110)
    df["rsrq_threshold_gap"]    = df[rsrq_col] - (-15)
    return df

print("✓ Feature engineering functions defined")

✓ Feature engineering functions defined


In [ ]:
# ── Apply All Feature Engineering ────────────────────────────────────────
df_feat = df.copy()
sig_cols = [rsrp_col, rsrq_col]
cols_before = df_feat.shape[1]

print("Generating features ...")
df_feat = add_lag_features(df_feat, sig_cols, [1,2,3])
df_feat = add_derivatives(df_feat, sig_cols)
df_feat = add_rolling_stats(df_feat, sig_cols, ROLL_WIN)
df_feat = add_degradation(df_feat, sig_cols)
df_feat = add_stability(df_feat, sig_cols, TREND_WIN)
df_feat = add_cross_features(df_feat, rsrp_col, rsrq_col)
df_feat = add_risk_features(df_feat, rsrp_col, rsrq_col)

rows_before = len(df_feat)
df_feat = df_feat.dropna(subset=[c for c in df_feat.columns if c not in sig_cols+[TARGET]])
print(f"  Dropped {rows_before-len(df_feat)} NaN rows")
print(f"\n✓ Features: {cols_before} → {df_feat.shape[1]} (+{df_feat.shape[1]-cols_before} new)")
print(f"  Dataset: {df_feat.shape[0]:,} rows × {df_feat.shape[1]} columns")

df_feat.to_csv("outputs/dataset_engineered.csv", index=False)
print("✓ Saved → outputs/dataset_engineered.csv")
df_feat.head(3)

Generating features ...
  Dropped 22 NaN rows

✓ Features: 19 → 60 (+41 new)
  Dataset: 11,071 rows × 60 columns
✓ Saved → outputs/dataset_engineered.csv


,deviceid,network_provi.,networktype,rsrq,downlink(mbps),velocity(km/h),uplink(mbps),devicemodel,pci,devicemake,...,rsrq_slope10,rsrp_rsrq_diff,rsrp_rsrq_ratio,signal_quality_index,both_dropping,rsrp_below_threshold,rsrq_below_threshold,dual_threshold_breach,rsrp_threshold_gap,rsrq_threshold_gap
20,65bb12ad684ceba4,airtel,WCDMA (3G),-12.0,8.0,32.94,3.0,SM-S901E,232.0,samsung,...,0.0,-12.0,2.0,-18.0,0,0,0,0,86.0,3.0
21,65bb12ad684ceba4,airtel,WCDMA (3G),-12.0,8.0,32.94,3.0,SM-S901E,232.0,samsung,...,0.0,-12.0,2.0,-18.0,0,0,0,0,86.0,3.0
22,65bb12ad684ceba4,airtel,WCDMA (3G),-12.0,8.0,32.94,3.0,SM-S901E,232.0,samsung,...,0.0,-12.0,2.0,-18.0,0,0,0,0,86.0,3.0


In [ ]:
# ── Stage 2 Plots ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10))
fig.suptitle("Stage 2 — Temporal Feature Engineering", fontsize=14, fontweight="bold")
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
n_plot = min(400, len(df_feat))
idx = np.arange(n_plot)

ax = fig.add_subplot(gs[0, :2])
ax.plot(idx, df_feat[rsrp_col].iloc[:n_plot].values, label=rsrp_col, color="#2c3e50", lw=1.0)
for lag in [1,2,3]:
    col = f"{rsrp_col}_lag{lag}"
    if col in df_feat.columns:
        ax.plot(idx, df_feat[col].iloc[:n_plot].values, label=col, alpha=0.6, lw=0.8, ls="--")
ax.set_title("RSRP + Lag Features", fontweight="bold"); ax.legend(fontsize=7); ax.grid(alpha=0.3)

ax2 = fig.add_subplot(gs[0, 2])
mean_c = f"{rsrp_col}_mean{ROLL_WIN}"; std_c = f"{rsrp_col}_std{ROLL_WIN}"
mu = df_feat[mean_c].iloc[:n_plot].values; sig = df_feat[std_c].iloc[:n_plot].values
ax2.plot(idx, mu, color="#3498db", label="Rolling Mean")
ax2.fill_between(idx, mu-sig, mu+sig, alpha=0.25, color="#3498db", label="±1 STD")
ax2.set_title(f"Rolling Stats (w={ROLL_WIN})", fontweight="bold"); ax2.legend(fontsize=8)

ax3 = fig.add_subplot(gs[1, 0])
diff_c = f"{rsrp_col}_diff"
ax3.plot(idx, df_feat[diff_c].iloc[:n_plot].values, color="#9b59b6", lw=0.7)
ax3.axhline(0, color="black", lw=0.8, ls="--")
ax3.set_title("RSRP Derivative", fontweight="bold"); ax3.grid(alpha=0.3)

ax4 = fig.add_subplot(gs[1, 1])
streak_c = f"{rsrp_col}_drop_streak"
ax4.plot(idx, df_feat[streak_c].iloc[:n_plot].values, color="#e74c3c", lw=0.8)
ax4.set_title("RSRP Drop Streak", fontweight="bold"); ax4.grid(alpha=0.3)

ax5 = fig.add_subplot(gs[1, 2])
num_c = df_feat.select_dtypes(include=np.number).columns.tolist()
corr_t = (df_feat[num_c].corr()[TARGET].drop(TARGET).dropna()
          .abs().sort_values(ascending=False).head(15))
colors_c = ["#e74c3c" if v > 0.3 else "#3498db" for v in corr_t.values]
ax5.barh(corr_t.index[::-1], corr_t.values[::-1], color=colors_c[::-1], edgecolor="black", lw=0.4)
ax5.set_title("Top-15 Feature Correlations with Target", fontweight="bold")

plt.savefig("plots/stage2_features.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Stage 2 complete")

✓ Stage 2 complete


---
## ⚖️ Stage 3 — Class Imbalance Handling

In [ ]:
# ── Prepare Feature Matrix ────────────────────────────────────────────────
# FIX 5: Filter non-numeric columns properly with is_numeric_dtype()
# FIX 6: TARGET stays as "handover" — original notebook overwrote it in cell 21

drop_cols = [c for c in df_feat.columns
             if not pd.api.types.is_numeric_dtype(df_feat[c])
             or "timestamp" in c.lower()
             or c.lower() == "time"]
df_model = df_feat.drop(columns=drop_cols, errors="ignore").dropna()

y_raw = df_model[TARGET].values.astype(int)
X_raw = df_model.drop(columns=[TARGET]).values.astype(np.float64)
clean_feature_names = df_model.drop(columns=[TARGET]).columns.tolist()

print(f"Dataset: {df_model.shape}  |  Features: {len(clean_feature_names)}")
classes, counts = np.unique(y_raw, return_counts=True)
print(f"Class distribution: {dict(zip(classes.tolist(), counts.tolist()))}")
print(f"Imbalance ratio: {counts.max()/counts.min():.1f}:1")

scaler = StandardScaler()
X_sc_clean = scaler.fit_transform(X_raw)

with open("models/feature_scaler.pkl","wb") as f: pickle.dump(scaler, f)
with open("models/feature_names.pkl","wb") as f: pickle.dump(clean_feature_names, f)

Dataset: (11071, 54)  |  Features: 53
Class distribution: {0: 10726, 1: 345}
Imbalance ratio: 31.1:1


In [ ]:
# ── Manual SMOTE Fallback ─────────────────────────────────────────────────
def manual_smote(X, y, k=5, ratio=1.0, random_state=42):
    rng = np.random.default_rng(random_state)
    classes, counts = np.unique(y, return_counts=True)
    minority_class  = classes[np.argmin(counts)]
    X_min = X[y == minority_class]
    target_samples  = int(counts.max() * ratio)
    needed = target_samples - len(X_min)

    if needed <= 0:
        return X, y

    synthetic = []
    for _ in range(needed):
        idx = rng.integers(0, len(X_min))
        sample = X_min[idx]

        dists = np.linalg.norm(X_min - sample, axis=1)
        dists[idx] = np.inf
        nn_idx = np.argsort(dists)[:k]

        nn = X_min[rng.choice(nn_idx)]
        synthetic.append(sample + rng.uniform(0,1,sample.shape) * (nn - sample))

    X_syn = np.vstack(synthetic)
    y_syn = np.full(len(X_syn), minority_class)

    return np.vstack([X, X_syn]), np.concatenate([y, y_syn])


# ── Strategy Evaluation with 30 Random Seeds ──────────────────────────────
# FIX: SMOTE applied INSIDE each fold + results averaged across seeds

def eval_strategy(X, y, name, n_splits=5, use_smote=False, smote_ratio=1.0, n_repeats=30):

    f1s, recs, precs, accs = [], [], [], []

    for seed in range(n_repeats):   # ← run experiment multiple times

        clf = RandomForestClassifier(
            n_estimators=80,
            max_depth=10,
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1
        )

        skf = StratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=seed
        )

        for tr_idx, va_idx in skf.split(X, y):

            X_tr, y_tr = X[tr_idx], y[tr_idx]
            X_va, y_va = X[va_idx], y[va_idx]

            # SMOTE applied only on training fold
            if use_smote and (y_tr == 1).sum() >= 5:

                if IMBLEARN:
                    try:
                        X_tr, y_tr = SMOTE(
                            random_state=seed,
                            k_neighbors=min(5, (y_tr==1).sum()-1)
                        ).fit_resample(X_tr, y_tr)

                    except:
                        X_tr, y_tr = manual_smote(
                            X_tr,
                            y_tr,
                            ratio=smote_ratio,
                            random_state=seed
                        )
                else:
                    X_tr, y_tr = manual_smote(
                        X_tr,
                        y_tr,
                        ratio=smote_ratio,
                        random_state=seed
                    )

            clf.fit(X_tr, y_tr)
            preds = clf.predict(X_va)

            f1s.append(f1_score(y_va, preds, zero_division=0))
            recs.append(recall_score(y_va, preds, zero_division=0))
            precs.append(precision_score(y_va, preds, zero_division=0))
            accs.append(accuracy_score(y_va, preds))


    res = {
        "strategy": name,
        "f1": float(np.mean(f1s)),
        "f1_std": float(np.std(f1s)),
        "recall": float(np.mean(recs)),
        "precision": float(np.mean(precs)),
        "accuracy": float(np.mean(accs))
    }

    print(f"  {name:<30} F1={res['f1']:.4f} ±{res['f1_std']:.4f}  Recall={res['recall']:.4f}  Acc={res['accuracy']:.4f}")

    return res


# ── Run Strategy Experiments ──────────────────────────────────────────────
configs = [
    ("No resampling", False, 1.0),
    ("SMOTE ratio 0.5", True, 0.5),
    ("SMOTE ratio 0.7", True, 0.7),
    ("SMOTE ratio 1.0", True, 1.0)
]

results = []

for name, use_smote, ratio in configs:
    results.append(
        eval_strategy(
            X_sc_clean,
            y_raw,
            name,
            use_smote=use_smote,
            smote_ratio=ratio
        )
    )

res_df = pd.DataFrame(results).sort_values("f1", ascending=False)

print("\nStrategy Ranking (by F1):")
print(res_df[["strategy","f1","f1_std","recall","precision","accuracy"]].to_string(index=False))


best_name  = res_df.iloc[0]["strategy"]
best_smote = [c[1] for c in configs if c[0]==best_name][0]
best_ratio = [c[2] for c in configs if c[0]==best_name][0]

print(f"\n✓ Best: '{best_name}'  F1={res_df.iloc[0]['f1']:.4f}")

  No resampling                  F1=0.6621 ±0.0345  Recall=0.8434  Acc=0.9731
  SMOTE ratio 0.5                F1=0.6700 ±0.0276  Recall=0.8751  Acc=0.9731
  SMOTE ratio 0.7                F1=0.6700 ±0.0276  Recall=0.8751  Acc=0.9731
  SMOTE ratio 1.0                F1=0.6700 ±0.0276  Recall=0.8751  Acc=0.9731

Strategy Ranking (by F1):
       strategy       f1   f1_std   recall  precision  accuracy
SMOTE ratio 0.5 0.669995 0.027623 0.875072   0.543799  0.973083
SMOTE ratio 0.7 0.669995 0.027623 0.875072   0.543799  0.973083
SMOTE ratio 1.0 0.669995 0.027623 0.875072   0.543799  0.973083
  No resampling 0.662068 0.034513 0.843382   0.546042  0.973104

✓ Best: 'SMOTE ratio 0.5'  F1=0.6700


In [ ]:
# ── Apply Best Strategy & Plot ────────────────────────────────────────────
# FIX 8: SMOTE applied only on TRAINING split (not entire dataset)

from imblearn.combine import SMOTETomek

X_tr_raw, X_te_ens, y_tr_raw, y_te_ens = train_test_split(
    X_sc_clean,
    y_raw,
    test_size=0.2,
    stratify=y_raw,
    random_state=RANDOM_STATE
)

y_tr_raw = y_tr_raw.astype(int)
y_te_ens = y_te_ens.astype(int)

# Apply Tomek Links cleaning + SMOTE oversampling
if (y_tr_raw == 1).sum() >= 5:
    X_tr_ens, y_tr_ens = SMOTETomek(random_state=RANDOM_STATE).fit_resample(
        X_tr_raw, y_tr_raw
    )
else:
    X_tr_ens, y_tr_ens = X_tr_raw, y_tr_raw

y_tr_ens = y_tr_ens.astype(int)

cls_b, cnt_b = np.unique(y_tr_ens, return_counts=True)
print(f"Train after resampling: {X_tr_ens.shape}  |  Classes: {dict(zip(cls_b.tolist(), cnt_b.tolist()))}")
print(f"Test: {X_te_ens.shape}")

# Stage 3 Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Stage 3 — Class Imbalance Handling", fontsize=14, fontweight="bold")

# Original distribution
classes_orig, counts_orig = np.unique(y_raw, return_counts=True)
axes[0].pie(
    counts_orig,
    labels=[f"Class {c}" for c in classes_orig],
    colors=["#2ecc71", "#e74c3c"],
    autopct="%1.1f%%",
    wedgeprops={"edgecolor": "white", "linewidth": 2},
)
axes[0].set_title("Original Distribution", fontweight="bold")

# After resampling
_, bc = np.unique(y_tr_ens, return_counts=True)
axes[1].pie(
    bc,
    labels=[f"Class {c}" for c in [0, 1]],
    colors=["#2ecc71", "#e74c3c"],
    autopct="%1.1f%%",
    wedgeprops={"edgecolor": "white", "linewidth": 2},
)
axes[1].set_title("After SMOTETomek (train only)", fontweight="bold")

# Strategy comparison plot
names_p = [r["strategy"] for r in results]
f1s_p = [r["f1"] for r in results]

bar_c = ["#e74c3c" if n == best_name else "#95a5a6" for n in names_p]

bars = axes[2].barh(names_p, f1s_p, color=bar_c, edgecolor="black", linewidth=0.6)

for bar, v in zip(bars, f1s_p):
    axes[2].text(v + 0.002, bar.get_y() + bar.get_height() / 2, f"{v:.3f}", va="center", fontsize=9)

axes[2].set_title("F1 by Strategy", fontweight="bold")
axes[2].set_xlabel("F1")

plt.tight_layout()
plt.savefig("plots/stage3_imbalance.png", dpi=150, bbox_inches="tight")
plt.show()

print("✓ Stage 3 complete")

Train after resampling: (17160, 53)  |  Classes: {0: 8580, 1: 8580}
Test: (2215, 53)
✓ Stage 3 complete


---
## 🤖 Stage 4 — AutoGluon / Stacked Ensemble Training

In [ ]:
# ── AutoGluon (if available) ──────────────────────────────────────────────
if AUTOGLUON:
    train_ag = pd.DataFrame(X_tr_ens, columns=clean_feature_names)
    train_ag[TARGET] = y_tr_ens
    test_ag  = pd.DataFrame(X_te_ens, columns=clean_feature_names)
    train_ag[TARGET] = train_ag[TARGET].astype(int).clip(0,1)

    predictor = TabularPredictor(label=TARGET, eval_metric="f1",
                                  path="models/autogluon_models").fit(
        train_data=train_ag, presets="high_quality",
        num_bag_folds=5, num_stack_levels=1,
        hyperparameters={"GBM":{},"CAT":{},"XGB":{},"RF":{}},
        time_limit=1800)

    y_pred_final = predictor.predict(test_ag).values
    proba = predictor.predict_proba(test_ag)
    y_proba_final = proba.iloc[:,1].values if proba.shape[1]==2 else proba.max(axis=1).values
    lb = predictor.leaderboard(silent=True)
    print("\nAutoGluon Leaderboard:"); print(lb.to_string(index=False))
    lb.to_csv("outputs/model_leaderboard.csv", index=False)
    print("\n✓ AutoGluon complete")

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          2
Pytorch Version:    2.10.0+cpu
CUDA Version:       CUDA is not available
Memory Avail:       9.86 GB / 12.67 GB (77.8%)
Disk Space Avail:   85.77 GB / 107.72 GB (79.6%)
Presets specified: ['high_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=5, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~5x), but runs the risk of an out-of-memory error during model refit if memory is small relative to the data size.
	You can avoid this risk by setting `save_bag_folds=True`.
DyStack is enabled (dynamic_stacking=Tr

KeyboardInterrupt: 

In [ ]:
# ── sklearn Stacked Ensemble (always runs; primary if no AutoGluon) ───────
print("Building sklearn stacked ensemble ...")

def get_base_estimators():
    est = [
        ("RandomForest", RandomForestClassifier(n_estimators=150, class_weight="balanced",
                                                 max_depth=12, random_state=RANDOM_STATE, n_jobs=-1)),
        ("ExtraTrees",   ExtraTreesClassifier(n_estimators=150, class_weight="balanced",
                                               max_depth=12, random_state=RANDOM_STATE, n_jobs=-1)),
        ("GBT",          GradientBoostingClassifier(n_estimators=80, learning_rate=0.1,
                                                     max_depth=5, random_state=RANDOM_STATE)),
        ("MLP",          MLPClassifier(hidden_layer_sizes=(128,64), activation="relu",
                                       solver="adam", max_iter=200, early_stopping=True,
                                       random_state=RANDOM_STATE)),
    ]
    if LGBM:
        est.insert(2,("LightGBM", lgb.LGBMClassifier(n_estimators=500, learning_rate=0.03,
                                                       num_leaves=63, class_weight="balanced",
                                                       random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)))
    if XGB:
        est.insert(3,("XGBoost", xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                                     eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1)))
    if CATBOOST:
        est.insert(4,("CatBoost", CatBoostClassifier(iterations=500, learning_rate=0.05,
                                                      depth=8, random_seed=RANDOM_STATE, verbose=0)))
    if len(X_tr_ens) <= 50000:
        est.append(("KNN", KNeighborsClassifier(n_neighbors=11, weights="distance", n_jobs=-1)))
    return est

base_ests = get_base_estimators()
n_models  = len(base_ests)
skf_ens   = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
oof_preds_mat  = np.zeros((len(X_tr_ens), n_models))
test_preds_mat = np.zeros((len(X_te_ens), n_models))
base_scores_ens = {}

for j, (name, clf) in enumerate(base_ests):
    print(f"  [{j+1}/{n_models}] {name:<15}", end="  ")
    t0 = time.time()
    fold_te = np.zeros((len(X_te_ens), N_FOLDS))
    for fold, (tri, vai) in enumerate(skf_ens.split(X_tr_ens, y_tr_ens)):
        clf.fit(X_tr_ens[tri], y_tr_ens[tri])
        if hasattr(clf, "predict_proba"):
            oof_preds_mat[vai,j] = clf.predict_proba(X_tr_ens[vai])[:,1]
            fold_te[:,fold]       = clf.predict_proba(X_te_ens)[:,1]
        else:
            oof_preds_mat[vai,j] = clf.predict(X_tr_ens[vai])
            fold_te[:,fold]       = clf.predict(X_te_ens)
    test_preds_mat[:,j] = fold_te.mean(axis=1)
    oof_f1_val = f1_score(y_tr_ens, (oof_preds_mat[:,j]>0.5).astype(int), zero_division=0)
    base_scores_ens[name] = oof_f1_val
    print(f"OOF F1={oof_f1_val:.4f}  ({time.time()-t0:.1f}s)")

meta = LogisticRegression(C=1.0, max_iter=500, class_weight="balanced", random_state=RANDOM_STATE)
meta.fit(oof_preds_mat, y_tr_ens)
y_pred_sklearn  = meta.predict(test_preds_mat)
y_proba_sklearn = meta.predict_proba(test_preds_mat)[:,1]

with open("models/base_estimators.pkl","wb") as f: pickle.dump(base_ests, f)
with open("models/meta_learner.pkl","wb") as f:    pickle.dump(meta, f)
print("\n✓ sklearn stacked ensemble trained and saved")

Building sklearn stacked ensemble ...
  [1/8] RandomForest     OOF F1=0.9885  (39.3s)
  [2/8] ExtraTrees       OOF F1=0.9824  (7.0s)
  [3/8] LightGBM         OOF F1=0.9934  (44.6s)
  [4/8] XGBoost          OOF F1=0.9926  (13.2s)
  [5/8] CatBoost         

In [ ]:
# ── Final Predictions & Metrics ──────────────────────────────────────────
if AUTOGLUON:
    print("Using AutoGluon predictions")
else:
    y_pred_final  = y_pred_sklearn
    y_proba_final = y_proba_sklearn
    lb_rows = [{"model": k, "oof_f1": v} for k,v in base_scores_ens.items()]
    lb_rows.append({"model":"StackedEnsemble",
                    "oof_f1": f1_score(y_te_ens, y_pred_final, zero_division=0)})
    lb = pd.DataFrame(lb_rows).sort_values("oof_f1", ascending=False)
    print("\nLeaderboard:"); print(lb.to_string(index=False))
    lb.to_csv("outputs/model_leaderboard.csv", index=False)
    print("Using sklearn stacked ensemble predictions")

acc_s4  = accuracy_score(y_te_ens, y_pred_final)
f1_s4 = f1_score(y_te_ens, y_pred_final, average='macro', zero_division=0)
prec_s4 = precision_score(y_te_ens, y_pred_final, zero_division=0)
rec_s4  = recall_score(y_te_ens, y_pred_final, zero_division=0)
auc_s4  = roc_auc_score(y_te_ens, y_proba_final)

print(f"\n{'═'*52}")
print(f"  Accuracy  : {acc_s4*100:.2f}%  (Paper: ~97%)")
print(f"  F1 Score  : {f1_s4:.4f}")
print(f"  Precision : {prec_s4:.4f}")
print(f"  Recall    : {rec_s4:.4f}")
print(f"  ROC-AUC   : {auc_s4:.4f}")
print(f"  {'✓ TARGET ACHIEVED (≥98%)' if acc_s4>=0.98 else '⚠  Below 98% target'}")
print(f"{'═'*52}")

pd.DataFrame([{"y_true":a,"y_pred":b,"y_proba":c}
              for a,b,c in zip(y_te_ens, y_pred_final, y_proba_final)]
             ).to_csv("outputs/stage4_predictions.csv", index=False)

Using AutoGluon predictions

════════════════════════════════════════════════════
  Accuracy  : 98.15%  (Paper: ~97%)
  F1 Score  : 0.6917
  Precision : 0.7188
  Recall    : 0.6667
  ROC-AUC   : 0.9914
  ✓ TARGET ACHIEVED (≥98%)
════════════════════════════════════════════════════


In [ ]:
# ── Stage 4 Plots ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10))
fig.suptitle("Stage 4 — Ensemble Training Results", fontsize=14, fontweight="bold")
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

ax1 = fig.add_subplot(gs[0,:2])
lb_plot = lb.head(10)
mod_names = lb_plot["model"].values
lb_vals   = lb_plot[lb_plot.columns[1]].values
colors_lb = ["#e74c3c" if i==0 else "#3498db" for i in range(len(lb_vals))]
bars = ax1.barh(mod_names[::-1], lb_vals[::-1], color=colors_lb[::-1], edgecolor="black", lw=0.5)
for bar, v in zip(bars, lb_vals[::-1]):
    ax1.text(v+0.002, bar.get_y()+bar.get_height()/2, f"{v:.4f}", va="center", fontsize=8)
ax1.set_title("Model Leaderboard", fontweight="bold"); ax1.set_xlabel("F1 Score")

ax2 = fig.add_subplot(gs[0, 2])
m_names = ["Accuracy","F1","Precision","Recall","AUC"]
ours_v  = [acc_s4, f1_s4, prec_s4, rec_s4, auc_s4]
paper_v = [0.97, 0.95, 0.96, 0.94, 0.97]
x = np.arange(len(m_names))
ax2.bar(x-0.2, ours_v,  0.35, label="Our Model", color="#e74c3c")
ax2.bar(x+0.2, paper_v, 0.35, label="Paper",     color="#95a5a6")
ax2.set_xticks(x); ax2.set_xticklabels(m_names, rotation=15, ha="right", fontsize=8)
ax2.set_ylim(0.85,1.0); ax2.set_title("vs Paper Baseline", fontweight="bold"); ax2.legend()

ax3 = fig.add_subplot(gs[1, 0])
ax3.hist(y_proba_final[y_te_ens==0], bins=40, alpha=0.7, color="#2ecc71", label="No HO", density=True)
ax3.hist(y_proba_final[y_te_ens==1], bins=40, alpha=0.7, color="#e74c3c", label="HO",    density=True)
ax3.axvline(0.5, color="black", lw=1.5, ls="--"); ax3.set_title("Probability Distribution", fontweight="bold")
ax3.legend(fontsize=8)

ax4 = fig.add_subplot(gs[1, 1])
fpr_s4, tpr_s4, _ = roc_curve(y_te_ens, y_proba_final)
ax4.plot(fpr_s4, tpr_s4, color="#e74c3c", lw=2, label=f"AUC={auc_s4:.4f}")
ax4.plot([0,1],[0,1],"k--",lw=0.8)
ax4.set_title("ROC Curve", fontweight="bold"); ax4.legend(); ax4.grid(alpha=0.3)

ax5 = fig.add_subplot(gs[1, 2]); ax5.axis("off")
summary = (f"STAGE 4 SUMMARY\n{'─'*35}\n"
           f"  Backend   : {'AutoGluon' if AUTOGLUON else 'sklearn Stack'}\n"
           f"  CV Folds  : {N_FOLDS}\n  Train     : {len(X_tr_ens):,}\n  Test      : {len(X_te_ens):,}\n\n"
           f"  Accuracy  : {acc_s4*100:.2f}%\n  F1 Score  : {f1_s4:.4f}\n"
           f"  Precision : {prec_s4:.4f}\n  Recall    : {rec_s4:.4f}\n  ROC-AUC   : {auc_s4:.4f}\n\n"
           f"  ≥98%: {'✓ YES' if acc_s4>=0.98 else '⚠ NO'}\n")
ax5.text(0.05, 0.95, summary, transform=ax5.transAxes, fontsize=9, va="top", fontfamily="monospace",
         bbox=dict(boxstyle="round,pad=0.5", facecolor="#ecf0f1",
                   edgecolor="#27ae60" if acc_s4>=0.98 else "#f39c12", linewidth=2))
plt.savefig("plots/stage4_ensemble.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Stage 4 complete")

✓ Stage 4 complete


---
## 📊 Stage 5 — Comprehensive Evaluation

In [ ]:
# ── Load predictions & compute all metrics ──────────────────────────────
preds_df   = pd.read_csv("outputs/stage4_predictions.csv")
y_true_s5  = preds_df["y_true"].values
y_pred_s5  = preds_df["y_pred"].values
y_proba_s5 = preds_df["y_proba"].values

acc_s5  = accuracy_score(y_true_s5, y_pred_s5)
f1_s5 = f1_score(y_true_s5, y_pred_s5, average='macro', zero_division=0)
prec_s5 = precision_score(y_true_s5, y_pred_s5, zero_division=0)
rec_s5  = recall_score(y_true_s5, y_pred_s5, zero_division=0)
auc_s5  = roc_auc_score(y_true_s5, y_proba_s5)
ap_s5   = average_precision_score(y_true_s5, y_proba_s5)
cm_s5   = confusion_matrix(y_true_s5, y_pred_s5)

print(classification_report(y_true_s5, y_pred_s5, target_names=["No HO","HO"]))
print("── vs Paper Baseline ──")
for metric, ours_v, paper_v in [
    ("Accuracy", acc_s5, 0.97), ("F1", f1_s5, 0.95),
    ("Precision", prec_s5, 0.96), ("Recall", rec_s5, 0.94), ("ROC-AUC", auc_s5, 0.97)
]:
    delta = (ours_v - paper_v)*100
    print(f"  {metric:<12}: {ours_v*100:.2f}%  vs  {paper_v*100:.2f}%  "
          f"{'▲' if delta>0 else '▼'} {abs(delta):.2f}%")

              precision    recall  f1-score   support

       No HO       0.99      0.99      0.99      2146
          HO       0.72      0.67      0.69        69

    accuracy                           0.98      2215
   macro avg       0.85      0.83      0.84      2215
weighted avg       0.98      0.98      0.98      2215

── vs Paper Baseline ──
  Accuracy    : 98.15%  vs  97.00%  ▲ 1.15%
  F1          : 69.17%  vs  95.00%  ▼ 25.83%
  Precision   : 71.88%  vs  96.00%  ▼ 24.12%
  Recall      : 66.67%  vs  94.00%  ▼ 27.33%
  ROC-AUC     : 99.14%  vs  97.00%  ▲ 2.14%


In [ ]:
# ── Evaluation Plots ──────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.suptitle("Stage 5 — Comprehensive Evaluation", fontsize=15, fontweight="bold")
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.38)

ax1 = fig.add_subplot(gs[0, 0])
im = ax1.imshow(cm_s5, cmap="Blues")
ax1.set_xticks([0,1]); ax1.set_yticks([0,1])
ax1.set_xticklabels(["No HO","HO"]); ax1.set_yticklabels(["No HO","HO"])
ax1.set_xlabel("Predicted"); ax1.set_ylabel("Actual")
ax1.set_title("Confusion Matrix", fontweight="bold")
for i in range(2):
    for j in range(2):
        c = "white" if cm_s5[i,j] > cm_s5.max()/2 else "black"
        ax1.text(j, i, f"{cm_s5[i,j]:,}", ha="center", va="center",
                 fontsize=13, fontweight="bold", color=c)
plt.colorbar(im, ax=ax1)

ax2 = fig.add_subplot(gs[0, 1])
fpr, tpr, _ = roc_curve(y_true_s5, y_proba_s5)
ax2.plot(fpr, tpr, color="#e74c3c", lw=2.5, label=f"Our Model (AUC={auc_s5:.4f})")
ax2.plot([0,1],[0,1],"k--",lw=0.8,label="Random")
ax2.fill_between(fpr,tpr,alpha=0.1,color="#e74c3c")
ax2.set_title("ROC Curve", fontweight="bold"); ax2.legend(); ax2.grid(alpha=0.3)

ax3 = fig.add_subplot(gs[0, 2])
prec_c, rec_c, _ = precision_recall_curve(y_true_s5, y_proba_s5)
ax3.plot(rec_c, prec_c, color="#9b59b6", lw=2.5, label=f"AP={ap_s5:.4f}")
ax3.axhline(y_true_s5.mean(), color="gray", ls="--", lw=1.2, label="Baseline")
ax3.set_title("Precision-Recall Curve", fontweight="bold"); ax3.legend()

ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(y_proba_s5[y_true_s5==0], bins=50, alpha=0.7, density=True, color="#2ecc71", label="No HO")
ax4.hist(y_proba_s5[y_true_s5==1], bins=50, alpha=0.7, density=True, color="#e74c3c", label="HO")
ax4.axvline(0.5, color="black", lw=1.5, ls="--"); ax4.set_title("Probability Distribution", fontweight="bold")
ax4.legend()

ax5 = fig.add_subplot(gs[1, 1])
m_labels = ["Accuracy","F1","Precision","Recall","AUC"]
ours_arr  = [acc_s5, f1_s5, prec_s5, rec_s5, auc_s5]
paper_arr = [0.97, 0.95, 0.96, 0.94, 0.97]
x = np.arange(len(m_labels))
b1 = ax5.bar(x-0.22, ours_arr,  0.4, color="#e74c3c", label="Our Model", edgecolor="black")
b2 = ax5.bar(x+0.22, paper_arr, 0.4, color="#95a5a6", label="Paper",     edgecolor="black")
for bar, v in zip(b1, ours_arr):
    ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003, f"{v:.3f}",
             ha="center", va="bottom", fontsize=7)
ax5.set_xticks(x); ax5.set_xticklabels(m_labels, rotation=15, ha="right")
ax5.set_ylim(0.85,1.02); ax5.set_title("vs Paper Baseline", fontweight="bold"); ax5.legend()

ax6 = fig.add_subplot(gs[1, 2])
tn, fp, fn, tp = cm_s5.ravel()
ax6.pie([tn,fp,fn,tp], labels=["True Neg","False Pos","False Neg","True Pos"],
        colors=["#2ecc71","#f39c12","#e74c3c","#3498db"],
        autopct="%1.1f%%", wedgeprops={"edgecolor":"white","linewidth":1.5})
ax6.set_title("Prediction Breakdown", fontweight="bold")

plt.savefig("plots/stage5_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Stage 5 complete | Accuracy: {acc_s5*100:.2f}%")

✓ Stage 5 complete | Accuracy: 98.15%


---
## 🔍 Stage 6 — Feature Importance & Explainability

In [ ]:
# ── Feature Importance via RandomForest ──────────────────────────────────
df_eng = pd.read_csv("outputs/dataset_engineered.csv")
drop_c = [c for c in df_eng.columns
          if not pd.api.types.is_numeric_dtype(df_eng[c]) or "timestamp" in c.lower()]
df_fi  = df_eng.drop(columns=drop_c, errors="ignore").dropna()
y_fi   = df_fi[TARGET].values.astype(int)
X_fi   = df_fi.drop(columns=[TARGET]).values.astype(np.float64)
fn_fi  = df_fi.drop(columns=[TARGET]).columns.tolist()

sc_fi   = StandardScaler()
X_fi_sc = sc_fi.fit_transform(X_fi)

rf_fi = RandomForestClassifier(n_estimators=150, n_jobs=-1,
                                class_weight="balanced", random_state=RANDOM_STATE)
rf_fi.fit(X_fi_sc, y_fi)
importances = rf_fi.feature_importances_
indices     = np.argsort(importances)[::-1]
top_n       = min(30, len(fn_fi))
top_idx     = indices[:top_n]
top_names   = [fn_fi[i] for i in top_idx]
top_imps    = importances[top_idx]

print("Top 15 Features:")
for r, (n, v) in enumerate(zip(top_names[:15], top_imps[:15]), 1):
    print(f"  {r:2d}. {n:<40} {v:.4f}  {'█'*int(v*300)}")

imp_df = pd.DataFrame({"feature":fn_fi,"importance":importances}
                      ).sort_values("importance",ascending=False)
imp_df.to_csv("outputs/feature_importance.csv", index=False)
print("\n✓ Saved → outputs/feature_importance.csv")

Top 15 Features:
   1. rsrp_diff_abs                            0.1681  ██████████████████████████████████████████████████
   2. rsrq_diff_abs                            0.1229  ████████████████████████████████████
   3. rsrp_std5                                0.1182  ███████████████████████████████████
   4. rsrp_range5                              0.0856  █████████████████████████
   5. rsrq_range5                              0.0622  ██████████████████
   6. rsrp_var10                               0.0611  ██████████████████
   7. rsrq_std5                                0.0534  ████████████████
   8. rsrq_var10                               0.0366  ██████████
   9. rsrp_diff                                0.0307  █████████
  10. rsrp_drop_rate                           0.0265  ███████
  11. rsrp_velocity                            0.0237  ███████
  12. rsrq_diff                                0.0174  █████
  13. rsrp_drop_streak                         0.0167  ████
  14. rsrq_velo

In [ ]:
# ── SHAP (if available) else ExtraTrees 2nd Opinion ─────────────────────
if SHAP:
    print("Computing SHAP values (500 rows sample) ...")
    X_samp    = X_fi_sc[:min(500, len(X_fi_sc))]
    explainer = shap.TreeExplainer(rf_fi)
    shap_vals = explainer.shap_values(X_samp)

    # Normalise to 2-D (samples × features)
    if isinstance(shap_vals, list):
        # Old SHAP: list of arrays, one per class → use positive class [1]
        shap_vals = shap_vals[1]
    elif isinstance(shap_vals, np.ndarray) and shap_vals.ndim == 3:
        # New SHAP: ndarray of shape (samples, features, classes)
        # Average absolute value across all classes → (samples, features)
        shap_vals = shap_vals.mean(axis=2)

    shap_mean = np.abs(shap_vals).mean(axis=0)      # guaranteed 1-D now
    shap_df   = pd.DataFrame({"feature":fn_fi,"mean_abs_shap":shap_mean}
                             ).sort_values("mean_abs_shap",ascending=False)
    shap_df.to_csv("outputs/shap_importance.csv", index=False)
    print("Top 10 SHAP features:")
    for _, row in shap_df.head(10).iterrows():
        print(f"  {row['feature']:<40} {row['mean_abs_shap']:.4f}")
else:
    print("SHAP not available — using ExtraTrees 2nd opinion")
    shap_vals = None

# ExtraTrees 2nd opinion (always computed)
et_fi = ExtraTreesClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
et_fi.fit(X_fi_sc, y_fi)
et_imp = et_fi.feature_importances_
top_et = np.argsort(et_imp)[::-1][:15]

Computing SHAP values (500 rows sample) ...
Top 10 SHAP features:
  rsrp_diff_abs                            0.0000
  sinr                                     0.0000
  velocity(km/h)                           0.0000
  rsrp_std5                                0.0000
  rsrq_diff_abs                            0.0000
  rsrq_velocity                            0.0000
  rsrq_diff                                0.0000
  rsrp_var10                               0.0000
  rsrq_std5                                0.0000
  rsrq_threshold_gap                       0.0000


In [ ]:
# ── Feature Importance Plots ──────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.suptitle("Stage 6 — Feature Importance & Explainability", fontsize=14, fontweight="bold")
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.38)

color_map = {"rsrp":"#e74c3c","rsrq":"#3498db","pred":"#2ecc71","lag":"#9b59b6",
             "diff":"#f39c12","mean":"#1abc9c","drop":"#c0392b","slope":"#8e44ad"}
bar_colors = []
for name in top_names:
    matched = "gray"
    for key, col in color_map.items():
        if key in name.lower(): matched = col; break
    bar_colors.append(matched)

ax1 = fig.add_subplot(gs[:, 0])
ax1.barh(top_names[::-1], top_imps[::-1], color=bar_colors[::-1], edgecolor="black", linewidth=0.4)
patches = [mpatches.Patch(color=c, label=k.upper()) for k,c in list(color_map.items())[:6]]
ax1.legend(handles=patches, fontsize=7, loc="lower right")
ax1.set_title(f"Top-{top_n} Feature Importances (RF)", fontweight="bold")
ax1.set_xlabel("Importance Score")

groups_imp = {
    "RSRP Base":    sum(importances[i] for i,n in enumerate(fn_fi) if "rsrp" in n and not any(x in n for x in ["lag","diff","mean"])),
    "RSRQ Base":    sum(importances[i] for i,n in enumerate(fn_fi) if "rsrq" in n and not any(x in n for x in ["lag","diff","mean"])),
    "Lag":          sum(importances[i] for i,n in enumerate(fn_fi) if "lag" in n),
    "Derivatives":  sum(importances[i] for i,n in enumerate(fn_fi) if "diff" in n or "velocity" in n),
    "Rolling":      sum(importances[i] for i,n in enumerate(fn_fi) if any(x in n for x in ["mean","std","min","max","range"])),
    "Degradation":  sum(importances[i] for i,n in enumerate(fn_fi) if "drop" in n or "streak" in n),
    "Stability":    sum(importances[i] for i,n in enumerate(fn_fi) if "var" in n or "slope" in n),
    "LSTM Preds":   sum(importances[i] for i,n in enumerate(fn_fi) if "pred" in n),
}
ax2 = fig.add_subplot(gs[0, 1])
ax2.barh(list(groups_imp.keys()), list(groups_imp.values()),
         color=plt.cm.Set3(np.linspace(0,1,len(groups_imp))), edgecolor="black", lw=0.5)
ax2.set_title("Feature Group Contributions", fontweight="bold"); ax2.set_xlabel("Total Importance")

ax3 = fig.add_subplot(gs[1, 1])
if SHAP and shap_vals is not None:
    # shap.summary_plot doesn't accept 'ax' — point it at ax3 via plt.sca()
    plt.sca(ax3)
    shap.summary_plot(shap_vals, X_samp, feature_names=fn_fi,
                      max_display=15, show=False, plot_type="bar")
    ax3.set_title("SHAP Mean |value|", fontweight="bold")
else:
    ax3.barh([fn_fi[i] for i in top_et][::-1], et_imp[top_et][::-1],
             color="#e67e22", edgecolor="black", lw=0.4)
    ax3.set_title("ExtraTrees Importance (2nd opinion)", fontweight="bold")
    ax3.set_xlabel("Importance")

plt.savefig("plots/stage6_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Stage 6 complete")

✓ Stage 6 complete


---
## 🧪 Stage 7 — Ablation Study

In [ ]:
# ── 4-Configuration Ablation Study ──────────────────────────────────────
all_features   = fn_fi
signal_feats   = [c for c in all_features
                  if any(c.startswith(x) for x in [rsrp_col, rsrq_col])
                  and not any(x in c for x in ["lag","diff","mean","std","min","max",
                                                "pred","drop","var","slope","streak",
                                                "velocity","range","ratio","index"])]
temporal_feats = [c for c in all_features if any(x in c for x in
                  ["lag","diff","velocity","mean","std","min","max","range",
                   "drop","streak","var","slope"])]
pred_feats     = [c for c in all_features if "pred" in c.lower()]

configurations = {
    "Config 1: Signal Only":                    signal_feats,
    "Config 2: Signal + Temporal":              signal_feats + temporal_feats,
    "Config 3: Signal + Temporal + Preds":      signal_feats + temporal_feats + pred_feats,
    "Config 4: Full Pipeline (All Features)":   all_features,
}

skf_abl = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
clf_abl = RandomForestClassifier(n_estimators=80, class_weight="balanced",
                                  random_state=RANDOM_STATE, n_jobs=-1)
sc_abl  = StandardScaler()

print("Running 5-fold CV for each configuration:\n")
ablation_results = []
for cfg_name, feat_list in configurations.items():
    avail = [f for f in feat_list if f in all_features] or all_features[:3]
    X_cfg    = df_fi[avail].values.astype(np.float64)
    X_sc_cfg = sc_abl.fit_transform(X_cfg)
    f1s, accs, recs, precs = [], [], [], []
    for tri, vai in skf_abl.split(X_sc_cfg, y_fi):
        clf_abl.fit(X_sc_cfg[tri], y_fi[tri])
        p = clf_abl.predict(X_sc_cfg[vai])
        f1s.append(f1_score(y_fi[vai], p, zero_division=0))
        accs.append(accuracy_score(y_fi[vai], p))
        recs.append(recall_score(y_fi[vai], p, zero_division=0))
        precs.append(precision_score(y_fi[vai], p, zero_division=0))
    res = {"configuration": cfg_name, "n_features": len(avail),
           "accuracy": np.mean(accs), "f1": np.mean(f1s),
           "f1_std": np.std(f1s), "recall": np.mean(recs), "precision": np.mean(precs)}
    ablation_results.append(res)
    print(f"  {cfg_name}")
    print(f"    n={len(avail):3d}  Acc={res['accuracy']:.4f}  "
          f"F1={res['f1']:.4f}±{res['f1_std']:.4f}  Recall={res['recall']:.4f}\n")

ab_df_final = pd.DataFrame(ablation_results)
ab_df_final.to_csv("outputs/ablation_results.csv", index=False)
print("✓ Ablation results saved")

Running 5-fold CV for each configuration:

  Config 1: Signal Only
    n=  6  Acc=0.8709  F1=0.1556±0.0120  Recall=0.3826

  Config 2: Signal + Temporal
    n= 41  Acc=0.9769  F1=0.5536±0.0314  Recall=0.4609

  Config 3: Signal + Temporal + Preds
    n= 44  Acc=0.9776  F1=0.5591±0.0312  Recall=0.4580

  Config 4: Full Pipeline (All Features)
    n= 53  Acc=0.9787  F1=0.5838±0.0482  Recall=0.4783

✓ Ablation results saved


In [ ]:
# ── Ablation Plots ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Stage 7 — Ablation Study: Feature Group Contributions",
             fontsize=14, fontweight="bold")
short_labels = [r["configuration"].split(":")[0] for r in ablation_results]
for ax, (metric_key, label, color) in zip(axes, [
    ("accuracy","Accuracy","#3498db"),("f1","F1 Score","#e74c3c"),("recall","Recall","#2ecc71")
]):
    vals = [r[metric_key] for r in ablation_results]
    stds = [r.get("f1_std",0) if metric_key=="f1" else 0 for r in ablation_results]
    x = np.arange(len(short_labels))
    bars = ax.bar(x, vals, color=color, edgecolor="black", linewidth=0.7, alpha=0.85)
    if metric_key=="f1":
        ax.errorbar(x, vals, yerr=stds, fmt="none", color="black", capsize=5, lw=1.5)
    best_idx = np.argmax(vals)
    bars[best_idx].set_edgecolor("#c0392b"); bars[best_idx].set_linewidth(2.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                f"{v:.4f}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(short_labels, rotation=12, ha="right", fontsize=9)
    ax.set_title(label, fontweight="bold")
    ax.set_ylim(max(0, min(vals)-0.05), min(1.0, max(vals)+0.04))
    ax.axhline(0.97, color="gray", lw=1.2, ls="--", label="Paper baseline")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("plots/stage7_ablation.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Stage 7 complete")

✓ Stage 7 complete


---
## 📋 Stage 8 — Final Report & Dashboard

In [ ]:
# ── Final Report ─────────────────────────────────────────────────────────
report = f"""
╔══════════════════════════════════════════════════════════════════╗
║     5G HANDOVER PREDICTION — RESEARCH PIPELINE FINAL REPORT     ║
╚══════════════════════════════════════════════════════════════════╝

FINAL TEST METRICS
──────────────────
  Metric         Our System      Paper (~97%)    Δ
  ─────────────────────────────────────────────────
  Accuracy       {acc_s5*100:.2f}%          97.00%        {(acc_s5-0.97)*100:+.2f}%
  F1 Score       {f1_s5:.4f}           0.9500          {(f1_s5-0.95)*100:+.2f}%
  Precision      {prec_s5:.4f}           0.9600          {(prec_s5-0.96)*100:+.2f}%
  Recall         {rec_s5:.4f}           0.9400          {(rec_s5-0.94)*100:+.2f}%
  ROC-AUC        {auc_s5:.4f}           0.9700          {(auc_s5-0.97)*100:+.2f}%

  Target ≥98%:  {'✓ ACHIEVED' if acc_s5>=0.98 else '⚠  NOT YET'}

ABLATION STUDY
──────────────
  Config                                    Features  Accuracy    F1"""

for _, row in ab_df_final.iterrows():
    report += f"\n  {row['configuration'][:42]:<44} {int(row['n_features']):3d}  {row['accuracy']*100:.2f}%  {row['f1']:.4f}"

report += "\n\nTOP-10 FEATURES\n───────────────"
for _, row in imp_df.head(10).iterrows():
    report += f"\n  {row['feature']:<40} {row['importance']:.4f}"

report += f"\n\n{'═'*65}"
print(report)
with open("outputs/final_report.txt","w") as f: f.write(report)
print("\n✓ Saved → outputs/final_report.txt")


╔══════════════════════════════════════════════════════════════════╗
║     5G HANDOVER PREDICTION — RESEARCH PIPELINE FINAL REPORT     ║
╚══════════════════════════════════════════════════════════════════╝

FINAL TEST METRICS
──────────────────
  Metric         Our System      Paper (~97%)    Δ
  ─────────────────────────────────────────────────
  Accuracy       98.15%          97.00%        +1.15%
  F1 Score       0.6917           0.9500          -25.83%
  Precision      0.7188           0.9600          -24.12%
  Recall         0.6667           0.9400          -27.33%
  ROC-AUC        0.9914           0.9700          +2.14%

  Target ≥98%:  ✓ ACHIEVED

ABLATION STUDY
──────────────
  Config                                    Features  Accuracy    F1
  Config 1: Signal Only                          6  87.09%  0.1556
  Config 2: Signal + Temporal                   41  97.69%  0.5536
  Config 3: Signal + Temporal + Preds           44  97.76%  0.5591
  Config 4: Full Pipeline (All Featur

In [ ]:
# ── Final Dashboard ───────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 8))
fig.suptitle("🏆 Final Results Dashboard — 5G Handover Prediction",
             fontsize=16, fontweight="bold")
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.5, wspace=0.45)

ax0 = fig.add_subplot(gs[0,:2]); ax0.axis("off")
color = "#27ae60" if acc_s5>=0.98 else "#e67e22"
ax0.text(0.5, 0.65, f"{acc_s5*100:.2f}%", ha="center", va="center",
         fontsize=58, fontweight="bold", color=color, transform=ax0.transAxes)
ax0.text(0.5, 0.25, f"{'✓ TARGET ACHIEVED (≥98%)' if acc_s5>=0.98 else '⚠  Below 98% target'}",
         ha="center", va="center", fontsize=14, color=color, transform=ax0.transAxes)
ax0.set_title("Final Accuracy", fontweight="bold", fontsize=13)
ax0.add_patch(plt.Rectangle((0.02,0.05),0.96,0.9,fill=False,
                              edgecolor=color, linewidth=3, transform=ax0.transAxes))

ax1 = fig.add_subplot(gs[0,2:])
m_labels = ["Accuracy","F1","Precision","Recall","AUC"]
ours_v   = [acc_s5, f1_s5, prec_s5, rec_s5, auc_s5]
paper_v  = [0.97, 0.95, 0.96, 0.94, 0.97]
x = np.arange(len(m_labels))
bars_ours  = ax1.bar(x-0.22, ours_v,  0.4, color="#e74c3c", label="Our Model", edgecolor="black")
bars_paper = ax1.bar(x+0.22, paper_v, 0.4, color="#bdc3c7", label="Paper",     edgecolor="black")
for bar, v in zip(bars_ours, ours_v):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
             f"{v:.3f}", ha="center", va="bottom", fontsize=8)
ax1.set_xticks(x); ax1.set_xticklabels(m_labels)
ax1.set_ylim(0.85,1.03); ax1.set_title("Our Model vs Paper", fontweight="bold")
ax1.legend(); ax1.grid(axis="y", alpha=0.3)

ax2 = fig.add_subplot(gs[1,0])
im = ax2.imshow(cm_s5, cmap="Blues")
ax2.set_xticks([0,1]); ax2.set_yticks([0,1])
ax2.set_xticklabels(["No HO","HO"]); ax2.set_yticklabels(["No HO","HO"])
for i in range(2):
    for j in range(2):
        c = "white" if cm_s5[i,j] > cm_s5.max()/2 else "black"
        ax2.text(j, i, f"{cm_s5[i,j]:,}", ha="center", va="center",
                 fontsize=11, fontweight="bold", color=c)
ax2.set_title("Confusion Matrix", fontweight="bold")

ax3 = fig.add_subplot(gs[1,1])
ax3.barh(top_names[:10][::-1], top_imps[:10][::-1], color="#3498db", edgecolor="black", lw=0.4)
ax3.set_title("Top-10 Features", fontweight="bold"); ax3.set_xlabel("Importance")

ax4 = fig.add_subplot(gs[1,2])
ab_labels = [r["configuration"].split(":")[0] for r in ablation_results]
ab_f1s    = [r["f1"] for r in ablation_results]
ab_colors = ["#e74c3c" if i==len(ab_labels)-1 else "#3498db" for i in range(len(ab_labels))]
ax4.bar(ab_labels, ab_f1s, color=ab_colors, edgecolor="black", lw=0.6)
ax4.set_title("Ablation: F1 by Config", fontweight="bold"); ax4.set_ylabel("F1")
ax4.set_xticklabels(ab_labels, rotation=20, ha="right", fontsize=8)

ax5 = fig.add_subplot(gs[1,3]); ax5.axis("off")
summary_txt = (f"SUMMARY\n{'─'*28}\n"
               f"  Accuracy  : {acc_s5*100:.2f}%\n  F1        : {f1_s5:.4f}\n"
               f"  Precision : {prec_s5:.4f}\n  Recall    : {rec_s5:.4f}\n"
               f"  ROC-AUC   : {auc_s5:.4f}\n\n"
               f"  Features  : {len(all_features)}\n"
               f"  Backend   : {'AutoGluon' if AUTOGLUON else 'sklearn Stack'}\n"
               f"  LSTM      : {'Yes' if TORCH else 'MLP fallback'}\n"
               f"  SMOTE     : {'imblearn' if IMBLEARN else 'Manual'}\n\n"
               f"  {'✓ TARGET MET' if acc_s5>=0.98 else '⚠ BELOW TARGET'}\n")
ax5.text(0.05, 0.95, summary_txt, transform=ax5.transAxes, fontsize=9, va="top",
         fontfamily="monospace",
         bbox=dict(boxstyle="round,pad=0.5", facecolor="#ecf0f1",
                   edgecolor="#27ae60" if acc_s5>=0.98 else "#e67e22", linewidth=2))

plt.savefig("plots/stage8_final_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n✅ ALL 8 STAGES COMPLETE!")
print(f"   Accuracy : {acc_s5*100:.2f}%  |  F1 : {f1_s5:.4f}  |  ROC-AUC : {auc_s5:.4f}")
print(f"   {'✓ TARGET ACHIEVED (≥98%)' if acc_s5>=0.98 else '⚠ Below 98% target'}")


✅ ALL 8 STAGES COMPLETE!
   Accuracy : 98.15%  |  F1 : 0.6917  |  ROC-AUC : 0.9914
   ✓ TARGET ACHIEVED (≥98%)


---
## 💾 Download All Outputs

In [ ]:
import shutil, zipfile
with zipfile.ZipFile("handover_pipeline_results.zip","w") as zf:
    for folder in ["outputs","plots","models"]:
        for root, _, files in os.walk(folder):
            for file in files:
                zf.write(os.path.join(root, file))
print("✓ Created handover_pipeline_results.zip")

try:
    from google.colab import files
    files.download("handover_pipeline_results.zip")
    print("✓ Download started")
except ImportError:
    print("Not in Colab — file saved as handover_pipeline_results.zip")

✓ Created handover_pipeline_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Download started
